# Get the DCNET model confidence for each question

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

In [21]:
# --------------------------------------------------
# Questions organized by RAVEN folder
# --------------------------------------------------

data_dict = {
    "center_single": [1228],
    "in_center_single_out_center_single": [219, 1229, 218],
    "up_center_single_down_center_single": [1228, 429],
    "left_center_single_right_center_single": [6569, 7108, 128],
    "in_distribute_four_out_center_single": [6999],
}


# --------------------------------------------------
# Device
# --------------------------------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


# --------------------------------------------------
# Load model only once
# --------------------------------------------------

model = DCNet().to(device)

model.load_state_dict(
    torch.load(
        "model_new_04.pth",
        map_location=device
    )
)

model.eval()

tf = ToTensor()

results = []


# --------------------------------------------------
# Run all folders and questions
# --------------------------------------------------

for folder_name, question_numbers in data_dict.items():

    print("\n" + "=" * 70)
    print(f"Folder: {folder_name}")
    print("=" * 70)

    for question_number in question_numbers:

        file_path = Path(
            "dataset",
            folder_name,
            f"RAVEN_{question_number}_test.npz"
        )

        print(f"\nQuestion: {question_number}")
        print(f"Path: {file_path}")

        # Skip missing files instead of stopping the entire loop
        if not file_path.exists():
            print("File not found — skipped.")
            continue

        try:
            data = np.load(file_path)

            images = data["image"]
            correct_answer = int(data["target"].item())

            # Shape expected by the model:
            # original: (16, 160, 160)
            # after ToTensor + batch dimension
            images_tensor = tf(images).unsqueeze(0).to(device)

            with torch.no_grad():
                output = model(images_tensor)

                probabilities_tensor = F.softmax(output, dim=1)

                predicted_answer = int(
                    torch.argmax(probabilities_tensor, dim=1).item()
                )

                confidence = float(
                    probabilities_tensor[0, predicted_answer].item()
                )

            probabilities = (
                probabilities_tensor
                .squeeze(0)
                .cpu()
                .numpy()
            )

            is_correct = predicted_answer == correct_answer

            print(f"Model predicted answer: {predicted_answer + 1}")
            print(f"Confidence: {confidence * 100:.1f}%")
            print(f"Correct answer: {correct_answer + 1}")
            print(f"Prediction correct: {is_correct}")

            print("\nProbabilities:")

            for option_index, probability in enumerate(
                probabilities,
                start=1
            ):
                print(
                    f"Choice {option_index}: "
                    f"{probability * 100:.1f}%"
                )

            print(
                f"Sum: {probabilities.sum() * 100:.2f}%"
            )

            # Save one row per question
            result_row = {
                "folder": folder_name,
                "question_number": question_number,
                "predicted_answer": predicted_answer + 1,
                "correct_answer": correct_answer + 1,
                "is_correct": is_correct,
                "confidence": confidence,
                "confidence_percent": confidence * 100,
            }

            # Save the probability of every possible answer
            for option_index, probability in enumerate(
                probabilities,
                start=1
            ):
                result_row[f"choice_{option_index}_probability"] = float(
                    probability
                )

            results.append(result_row)

        except Exception as error:
            print(f"Error processing question: {error}")

        finally:
            # Close the npz file
            if "data" in locals():
                data.close()

        print("-" * 70)


# --------------------------------------------------
# Convert results to a table
# --------------------------------------------------

results_df = pd.DataFrame(results)

print("\nFinal results:")
display(results_df)

Using device: mps

Folder: center_single

Question: 1228
Path: dataset/center_single/RAVEN_1228_test.npz
Model predicted answer: 2
Confidence: 66.0%
Correct answer: 2
Prediction correct: True

Probabilities:
Choice 1: 4.2%
Choice 2: 66.0%
Choice 3: 0.1%
Choice 4: 14.4%
Choice 5: 0.0%
Choice 6: 0.0%
Choice 7: 3.5%
Choice 8: 11.7%
Sum: 100.00%
----------------------------------------------------------------------

Folder: in_center_single_out_center_single

Question: 219
Path: dataset/in_center_single_out_center_single/RAVEN_219_test.npz
Model predicted answer: 2
Confidence: 44.2%
Correct answer: 2
Prediction correct: True

Probabilities:
Choice 1: 1.5%
Choice 2: 44.2%
Choice 3: 2.8%
Choice 4: 9.2%
Choice 5: 0.0%
Choice 6: 8.4%
Choice 7: 21.7%
Choice 8: 12.2%
Sum: 100.00%
----------------------------------------------------------------------

Question: 1229
Path: dataset/in_center_single_out_center_single/RAVEN_1229_test.npz
Model predicted answer: 7
Confidence: 79.4%
Correct answer: 7
P

,folder,question_number,predicted_answer,correct_answer,is_correct,confidence,confidence_percent,choice_1_probability,choice_2_probability,choice_3_probability,choice_4_probability,choice_5_probability,choice_6_probability,choice_7_probability,choice_8_probability
0,center_single,1228,2,2,True,0.660378,66.037786,0.042169,6.603779e-01,0.001288,1.443321e-01,1.148680e-13,1.067000e-06,0.035189,0.116643
1,in_center_single_out_center_single,219,2,2,True,0.441549,44.154885,0.015126,4.415489e-01,0.028295,9.158423e-02,1.270548e-17,8.368371e-02,0.217268,0.122495
2,in_center_single_out_center_single,1229,7,7,True,0.794155,79.415518,0.052533,3.760936e-02,0.000002,6.704812e-09,7.299627e-02,5.592329e-06,0.794155,0.042698
3,in_center_single_out_center_single,218,5,5,True,0.339196,33.919621,0.021881,2.223307e-01,0.193608,9.720163e-02,3.391962e-01,1.923053e-11,0.023771,0.102012
4,up_center_single_down_center_single,1228,1,1,True,0.285430,28.543007,0.285430,3.937517e-02,0.054041,2.002674e-01,2.452067e-01,8.451115e-02,0.001975,0.089194
5,up_center_single_down_center_single,429,8,7,False,0.265060,26.505962,0.009210,2.165296e-01,0.047979,1.789512e-01,9.084857e-02,9.176592e-07,0.191422,0.265060
6,left_center_single_right_center_single,6569,2,1,False,0.416765,41.676530,0.328257,4.167653e-01,0.001315,1.036469e-02,6.886603e-03,1.133846e-02,0.000548,0.224525
7,left_center_single_right_center_single,7108,7,6,False,0.466640,46.663985,0.062734,3.392695e-04,0.048763,2.299612e-04,1.074607e-04,3.641636e-01,0.466640,0.057023
8,left_center_single_right_center_single,128,7,8,False,0.345476,34.547639,0.000281,6.090591e-04,0.341080,7.369677e-03,5.776124e-03,2.803435e-02,0.345476,0.271373
9,in_distribute_four_out_center_single,6999,4,4,True,0.903402,90.340203,0.012833,3.187113e-09,0.015469,9.034020e-01,1.234519e-10,3.476237e-08,0.062806,0.005489
